<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/main/notebooks/pi05_libero_colab_l4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pi0.5 LIBERO Colab L4 Notebook

This notebook runs `lerobot/pi05_libero_finetuned` on LIBERO in Google Colab without Docker.

Target workflow:

1. Open this notebook in Colab.
2. Runtime -> Change runtime type -> GPU -> choose **L4** when available.
3. Mount a restricted Google Drive folder containing model/assets cache archives and experiment outputs.
4. Runtime -> Run all.
5. Inspect rollout videos and activation diagnostics inline.

Access policy to apply in Google Drive/Colab:

- Share this notebook only with approved Google accounts.
- Current requested approved account: `programmer908@gmail.com`.
- Share the Drive cache/output folder only with the same approved accounts.
- Do not store Hugging Face tokens in the notebook or shared Drive folder.
- Put `HF_TOKEN` only in each user's private Colab Secrets if an authenticated cache refresh is needed.

Performance note: Google Drive is slow for huge model caches with many files. This notebook uses tar archives under `DRIVE_ROOT/archives/` and extracts them to `/content` local disk before running.

Codex cannot enforce Google Drive sharing ACLs from this repo. Apply those restrictions in the Colab/Drive UI after saving the notebook copy.

## What This Runs

The standard benchmark path uses LeRobot eval:

```text
LIBERO simulator -> 2 camera tensors + robot state + task text -> Pi0.5 -> [50, 7] action chunk -> LIBERO simulator
```

The model receives two meaningful camera views on each policy call:

```text
observation.images.image   # agent view
observation.images.image2  # wrist / eye-in-hand view
```

It predicts 50 future actions. LeRobot executes the first 10 actions, then gets fresh camera/state observations and replans.

The display section creates an inline report for mechanistic interpretability: rollout video, four-panel diagnostic video, chunk matrix, family activation heatmaps, and per-expert-layer activation graphs. The final section includes an experimental single-chunk prompt probe. Benchmark scoring still uses official LIBERO task prompts and success predicates.

In [ ]:
# @title Controls

# Drive folder shared only with approved collaborators.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
APPROVED_ACCOUNTS = "programmer908@gmail.com"  # @param {type:"string"}

# Repository.
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "main"  # @param {type:"string"}

# Runtime preference. Use Any if Colab allocated a T4; set L4/A100 to enforce that GPU.
REQUIRED_GPU = "Any"  # @param ["L4", "A100", "Any"]

# Evaluation controls.
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10", "libero_spatial,libero_object,libero_goal,libero_10"]
TASK_IDS = "[0]"  # @param {type:"string"}
ANALYSIS_TASK_ID = 0  # @param {type:"integer"}
EPISODES = 1  # @param {type:"integer"}

# Activation capture controls.
CAPTURE_ACTIVATIONS = True  # @param {type:"boolean"}
CAPTURE_PARAM_STATS = False  # @param {type:"boolean"}
CAPTURE_MAX_CHUNKS = 40  # @param {type:"integer"}
CAPTURE_LAYER_STRIDE = 1  # @param {type:"integer"}
CAPTURE_MAX_BINS = 64  # @param {type:"integer"}

# Colab report controls.
REPORT_MAX_ROWS = 80  # @param {type:"integer"}
GENERATE_DIAGNOSTIC_VIDEO = False  # @param {type:"boolean"}
DISPLAY_INDIVIDUAL_LAYER_GRAPHS = False  # @param {type:"boolean"}
LAYER_GRAPH_LIMIT = 6  # @param {type:"integer"}

# Cache behavior. Archive mode avoids slow Google Drive small-file copies.
CACHE_TRANSFER_MODE = "archive"  # @param ["archive", "folders"]
ALLOW_AUTH_REFRESH = True  # @param {type:"boolean"}
FORCE_AUTH_REFRESH = False  # @param {type:"boolean"}
HF_OFFLINE = True  # @param {type:"boolean"}

# Experimental prompt probe controls.
PROBE_SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
PROBE_TASK_ID = 0  # @param {type:"integer"}
PROBE_LANGUAGE = "pick up the black bowl between the plate and the ramekin and place it on the plate"  # @param {type:"string"}
PROBE_SEED = 1000  # @param {type:"integer"}

print("Configured approved accounts:", APPROVED_ACCOUNTS)

In [ ]:
# @title Mount Drive And Validate Runtime

from pathlib import Path
import os
import subprocess

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path(DRIVE_ROOT)
DRIVE_ARCHIVES = DRIVE_ROOT / "archives"
DRIVE_HF_HOME = DRIVE_ROOT / "hf_home"
DRIVE_LIBERO_CACHE = DRIVE_ROOT / "libero_cache"
DRIVE_LIBERO_DATASETS = DRIVE_ROOT / "libero_datasets"
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
DRIVE_NOTEBOOK_META = DRIVE_ROOT / "notebook_meta"
for path in [DRIVE_ROOT, DRIVE_ARCHIVES, DRIVE_HF_HOME, DRIVE_LIBERO_CACHE, DRIVE_LIBERO_DATASETS, DRIVE_OUTPUTS, DRIVE_NOTEBOOK_META]:
    path.mkdir(parents=True, exist_ok=True)

HF_HOME_ARCHIVE = DRIVE_ARCHIVES / "hf_home.tar"
LIBERO_CACHE_ARCHIVE = DRIVE_ARCHIVES / "libero_cache.tar"
LIBERO_DATASETS_ARCHIVE = DRIVE_ARCHIVES / "libero_datasets.tar"

print("Drive root:", DRIVE_ROOT)
print("Archive dir:", DRIVE_ARCHIVES)
print("Expected sharing: restricted to", APPROVED_ACCOUNTS)

result = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True, capture_output=True)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise RuntimeError("No NVIDIA GPU visible. In Colab: Runtime -> Change runtime type -> GPU.")

gpu_line = result.stdout.strip().splitlines()[0]
if REQUIRED_GPU != "Any" and REQUIRED_GPU.lower() not in gpu_line.lower():
    raise RuntimeError(f"Requested {REQUIRED_GPU}, but Colab allocated: {gpu_line}. Change runtime or set REQUIRED_GPU='Any'.")

In [ ]:
# @title Install Native Runtime Equivalent To cloud/libero/Dockerfile

import os
import subprocess
from pathlib import Path

VENV = Path("/content/lerobot-venv")
PYTHON = VENV / "bin/python"

apt_packages = [
    "build-essential", "ca-certificates", "cmake", "curl", "ffmpeg", "git", "pv", "rsync",
    "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
    "libosmesa6-dev", "libsm6", "libxext6", "libxrender1", "pkg-config",
]
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", *apt_packages], check=True)
subprocess.run(["python3", "-m", "pip", "install", "-q", "-U", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.12"], check=True)
subprocess.run(["uv", "venv", str(VENV), "--python", "3.12"], check=True)
subprocess.run([
    "uv", "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
    "lerobot[evaluation,libero,pi]", "hf-transfer", "opencv-python", "numpy",
], check=True)

os.environ["PATH"] = f"{VENV / 'bin'}:" + os.environ["PATH"]
print("Python:", PYTHON)
subprocess.run([str(PYTHON), "-c", "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"], check=True)

In [ ]:
# @title Clone Or Update Repo

from pathlib import Path
import subprocess

LOCAL_REPO = Path("/content/groot-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)

print("Repo:", LOCAL_REPO)
subprocess.run(["git", "-C", str(LOCAL_REPO), "rev-parse", "--short", "HEAD"], check=True)

In [ ]:
# @title Prepare Drive Caches And Offline Mode

from pathlib import Path
import os
import shutil
import shlex
import subprocess
import time

LOCAL_HF_HOME = Path("/content/hf_home")
LOCAL_LIBERO_CACHE = Path.home() / ".cache/libero"
LOCAL_OUTPUT_ROOT = LOCAL_REPO / "outputs/eval/pi05_libero"
LOCAL_DATA_ROOT = LOCAL_REPO / "data/libero/datasets"
LOCAL_LIBERO_CONFIG = LOCAL_REPO / ".libero"
for path in [LOCAL_OUTPUT_ROOT, LOCAL_LIBERO_CONFIG]:
    path.mkdir(parents=True, exist_ok=True)

repos = ["lerobot/pi05_libero_finetuned", "google/paligemma-3b-pt-224"]


def timed(label, fn):
    print(f"\n== {label} ==", flush=True)
    start = time.time()
    result = fn()
    print(f"== done: {label} in {time.time() - start:.1f}s ==", flush=True)
    return result


def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    return subprocess.run([str(x) for x in cmd], check=True, **kwargs)


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def dir_has_anything(path: Path) -> bool:
    return path.exists() and any(path.iterdir())


def hf_cache_has(repo_id: str, root: Path) -> bool:
    namespace, name = repo_id.split("/", 1)
    return (root / "hub" / f"models--{namespace}--{name}").exists()


def du(path: Path):
    if path.exists():
        run(["du", "-sh", path])


def extract_tar(archive: Path, dst: Path):
    reset_dir(dst)
    size = archive.stat().st_size
    cmd = f"set -euo pipefail; pv -petraf -s {size} {shlex.quote(str(archive))} | tar -C {shlex.quote(str(dst))} -xf -"
    run(["bash", "-lc", cmd])
    du(dst)


def create_tar(src: Path, archive: Path):
    if not dir_has_anything(src):
        print(f"Skipping archive for empty directory: {src}", flush=True)
        return
    archive.parent.mkdir(parents=True, exist_ok=True)
    tmp = archive.with_suffix(archive.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    size_text = subprocess.check_output(["du", "-sb", str(src)], text=True).split()[0]
    cmd = f"set -euo pipefail; tar -C {shlex.quote(str(src))} -cf - . | pv -petraf -s {size_text} > {shlex.quote(str(tmp))}"
    run(["bash", "-lc", cmd])
    tmp.replace(archive)
    run(["ls", "-lh", archive])


def rsync_tree(src: Path, dst: Path, reset: bool = True):
    src.mkdir(parents=True, exist_ok=True)
    if reset:
        reset_dir(dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
    run(["rsync", "-a", "--info=progress2", f"{src}/", f"{dst}/"])
    du(dst)


if CACHE_TRANSFER_MODE == "archive" and HF_HOME_ARCHIVE.exists() and not FORCE_AUTH_REFRESH:
    timed("extract HF cache archive from Drive to /content", lambda: extract_tar(HF_HOME_ARCHIVE, LOCAL_HF_HOME))
elif ALLOW_AUTH_REFRESH:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("ALLOW_AUTH_REFRESH=True requires a private Colab Secret named HF_TOKEN, unless DRIVE_ROOT/archives/hf_home.tar already exists.")
    reset_dir(LOCAL_HF_HOME)
    run([str(PYTHON), "-c", "import huggingface_hub; print('huggingface_hub OK')"])
    refresh_code = """
from huggingface_hub import snapshot_download
import os
repos = os.environ['REFRESH_REPOS'].split(',')
for repo in repos:
    print('refreshing', repo, flush=True)
    path = snapshot_download(repo_id=repo, cache_dir=os.environ['LOCAL_HF_HUB_CACHE'], token=os.environ['HF_TOKEN'], max_workers=8)
    print(repo, '->', path, flush=True)
"""
    env = os.environ.copy()
    for key in ["HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]:
        env.pop(key, None)
    env.update({
        "HF_TOKEN": token,
        "LOCAL_HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
        "REFRESH_REPOS": ",".join(repos),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "HF_XET_HIGH_PERFORMANCE": "1",
    })
    timed("download gated HF models to fast local disk", lambda: subprocess.run([str(PYTHON), "-c", refresh_code], env=env, check=True))
elif CACHE_TRANSFER_MODE == "folders" or dir_has_anything(DRIVE_HF_HOME):
    print("Archive missing; falling back to Drive folder rsync. This can be very slow for HF caches.", flush=True)
    timed("copy HF cache folder from Drive to /content", lambda: rsync_tree(DRIVE_HF_HOME, LOCAL_HF_HOME))
else:
    raise RuntimeError(
        "No HF cache archive or folder found in Drive. Set ALLOW_AUTH_REFRESH=True once with private Colab Secret HF_TOKEN."
    )

missing_local = [repo for repo in repos if not hf_cache_has(repo, LOCAL_HF_HOME)]
if missing_local:
    raise RuntimeError(
        "Local HF cache is missing: " + ", ".join(missing_local) + "\n"
        "Run once with ALLOW_AUTH_REFRESH=True, or populate DRIVE_ROOT/archives/hf_home.tar."
    )

if CACHE_TRANSFER_MODE == "archive" and LIBERO_CACHE_ARCHIVE.exists():
    timed("extract LIBERO cache archive from Drive", lambda: extract_tar(LIBERO_CACHE_ARCHIVE, LOCAL_LIBERO_CACHE))
else:
    timed("copy LIBERO cache folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_CACHE, LOCAL_LIBERO_CACHE))

if CACHE_TRANSFER_MODE == "archive" and LIBERO_DATASETS_ARCHIVE.exists():
    timed("extract LIBERO datasets archive from Drive", lambda: extract_tar(LIBERO_DATASETS_ARCHIVE, LOCAL_DATA_ROOT))
else:
    timed("copy LIBERO datasets folder from Drive", lambda: rsync_tree(DRIVE_LIBERO_DATASETS, LOCAL_DATA_ROOT))

os.environ.update({
    "PATH": f"{VENV / 'bin'}:" + os.environ["PATH"],
    "PYTHON": str(PYTHON),
    "HF_HOME": str(LOCAL_HF_HOME),
    "HF_HUB_CACHE": str(LOCAL_HF_HOME / "hub"),
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "HF_XET_HIGH_PERFORMANCE": "1",
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "MUJOCO_EGL_DEVICE_ID": "0",
    "LIBERO_CONFIG_PATH": str(LOCAL_LIBERO_CONFIG),
    "LIBERO_DATASET_DIR": str(LOCAL_DATA_ROOT),
    "OUTPUT_ROOT": str(LOCAL_OUTPUT_ROOT),
})
if HF_OFFLINE:
    os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"})
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
    os.environ.pop("HF_DATASETS_OFFLINE", None)

validate_code = """
from huggingface_hub import snapshot_download
import os
for repo in os.environ['VALIDATE_REPOS'].split(','):
    path = snapshot_download(repo_id=repo, cache_dir=os.environ['HF_HUB_CACHE'], local_files_only=True)
    print(repo, '->', path, flush=True)
"""
env = os.environ.copy()
env["VALIDATE_REPOS"] = ",".join(repos)
timed("validate local HF cache with local_files_only=True", lambda: subprocess.run([str(PYTHON), "-c", validate_code], env=env, check=True))

print("\nLocal cache summary:")
du(LOCAL_HF_HOME)
du(LOCAL_LIBERO_CACHE)
du(LOCAL_DATA_ROOT)

In [ ]:
# @title Run LIBERO Eval

import os
import subprocess
from pathlib import Path

run_env = os.environ.copy()
run_env.update({
    "PYTHON": str(PYTHON),
    "DEVICE": "cuda",
    "DTYPE": "bfloat16",
    "TASK_IDS": TASK_IDS.strip(),
    "CAPTURE_ACTIVATIONS": "1" if CAPTURE_ACTIVATIONS else "0",
    "CAPTURE_PARAM_STATS": "1" if CAPTURE_PARAM_STATS else "0",
    "CAPTURE_MAX_CHUNKS": str(CAPTURE_MAX_CHUNKS),
    "CAPTURE_LAYER_STRIDE": str(CAPTURE_LAYER_STRIDE),
    "CAPTURE_MAX_BINS": str(CAPTURE_MAX_BINS),
})

cmd = ["bash", "cloud/libero/run_pi05_libero.sh", SUITE, str(EPISODES)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=LOCAL_REPO, env=run_env, check=True)

runs = sorted(p for p in LOCAL_OUTPUT_ROOT.glob("*") if p.is_dir())
if not runs:
    raise RuntimeError("No run directory produced")
LATEST_RUN = runs[-1]
print("Latest run:", LATEST_RUN)

In [ ]:
# @title Persist Outputs And Caches Back To Drive

# Outputs are small enough to sync as folders and should be visible directly in Drive.
rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)

# Persist caches as tar archives to avoid slow Drive small-file copies on the next Colab session.
if CACHE_TRANSFER_MODE == "archive":
    if FORCE_AUTH_REFRESH or not HF_HOME_ARCHIVE.exists():
        timed("write HF cache archive to Drive", lambda: create_tar(LOCAL_HF_HOME, HF_HOME_ARCHIVE))
    timed("write LIBERO cache archive to Drive", lambda: create_tar(LOCAL_LIBERO_CACHE, LIBERO_CACHE_ARCHIVE))
    timed("write LIBERO datasets archive to Drive", lambda: create_tar(LOCAL_DATA_ROOT, LIBERO_DATASETS_ARCHIVE))
else:
    timed("sync unpacked HF cache to Drive", lambda: rsync_tree(LOCAL_HF_HOME, DRIVE_HF_HOME))
    timed("sync unpacked LIBERO cache to Drive", lambda: rsync_tree(LOCAL_LIBERO_CACHE, DRIVE_LIBERO_CACHE))
    timed("sync unpacked LIBERO datasets to Drive", lambda: rsync_tree(LOCAL_DATA_ROOT, DRIVE_LIBERO_DATASETS))

print("Persisted outputs to:", DRIVE_OUTPUTS)
print("Cache archive dir:", DRIVE_ARCHIVES)

In [ ]:
# @title Display Summary, Videos, And Activation Report Inline

import json
import subprocess
from pathlib import Path
from IPython.display import display, Video, Markdown, Image, HTML

run_dir = LATEST_RUN
info_path = run_dir / "eval_info.json"
if info_path.exists():
    info = json.loads(info_path.read_text())
    overall = info.get("overall", {})
else:
    info = {}
    overall = {}

display(Markdown(f"""
### Latest run

`{run_dir}`

- success: `{overall.get('pc_success', 'n/a')}`
- episodes: `{overall.get('n_episodes', 'n/a')}`
- eval seconds: `{overall.get('eval_s', 'n/a')}`
"""))

videos = sorted((run_dir / "videos").glob("**/*.mp4"))
print("rollout videos:", len(videos))
if videos:
    display(Video(str(videos[0]), embed=True, width=720))

if CAPTURE_ACTIVATIONS:
    analysis_dir = run_dir / "analysis"
    analysis_dir.mkdir(parents=True, exist_ok=True)
    if GENERATE_DIAGNOSTIC_VIDEO:
        analysis_cmd = [
            str(PYTHON), "scripts/make_pi05_analysis_video.py",
            "--run", str(run_dir),
            "--task-id", str(ANALYSIS_TASK_ID),
            "--preview-frame", "30",
        ]
        subprocess.run(analysis_cmd, cwd=LOCAL_REPO, check=True)
        previews = sorted(analysis_dir.glob("*_frame0030.png"))
        analysis_videos = sorted(analysis_dir.glob("*.mp4"))
        if previews:
            display(Markdown("### Four-panel diagnostic preview"))
            display(Image(filename=str(previews[-1]), width=1000))
        if analysis_videos:
            display(Video(str(analysis_videos[-1]), embed=True, width=900))

    report_cmd = [
        str(PYTHON), "scripts/make_pi05_colab_report.py",
        "--run", str(run_dir),
        "--task-id", str(ANALYSIS_TASK_ID),
        "--episode", "0",
        "--n-action-steps", "10",
        "--max-rows", str(REPORT_MAX_ROWS),
    ]
    subprocess.run(report_cmd, cwd=LOCAL_REPO, check=True)
    report_dir = analysis_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_colab_report"
    manifest_path = report_dir / f"task_{ANALYSIS_TASK_ID}_episode_0_manifest.json"
    manifest = json.loads(manifest_path.read_text())

    display(Markdown("""
### Granular Investigation Report

The interactive report lets you switch between activation family, metric, and layer without rendering every layer image inline. The chunk matrix still gives one row per policy call: simulator third-person frame, the two camera tensors fed to Pi0.5, first 10 actions, expert-layer mean, and expert layer-by-denoise activation.
"""))
    interactive_path = manifest.get("interactive_html")
    if interactive_path and Path(interactive_path).exists():
        display(HTML(Path(interactive_path).read_text()))
    else:
        for key, width in [
            ("chunk_matrix", 1600),
            ("family_heatmaps", 1100),
            ("expert_layers_grid", 1100),
        ]:
            path = manifest.get(key)
            if path and Path(path).exists():
                display(Markdown(f"#### {key.replace('_', ' ').title()}"))
                display(Image(filename=str(path), width=width))

    layer_graphs = [Path(p) for p in manifest.get("expert_layer_graphs", [])]
    if DISPLAY_INDIVIDUAL_LAYER_GRAPHS and layer_graphs:
        display(Markdown("#### Individual Expert Layer Graphs"))
        for path in layer_graphs[:max(0, int(LAYER_GRAPH_LIMIT))]:
            if path.exists():
                display(Image(filename=str(path), width=900))

    rsync_tree(LOCAL_REPO / "outputs", DRIVE_OUTPUTS, reset=False)
else:
    display(Markdown("Activation report skipped because `CAPTURE_ACTIVATIONS=False`."))


## Experimental: One-Chunk Plaintext Prompt Probe

This cell is for mechanistic interpretability, not benchmark scoring.

It resets one LIBERO task environment, captures the two camera views and robot state, replaces the task language with `PROBE_LANGUAGE`, and asks Pi0.5 for one 50-action chunk. It saves the input images and action chunk so you can compare plaintext prompts against internal activations and action outputs.

Caveat: Pi0.5-LIBERO was fine-tuned on LIBERO-style task prompts. Low-level prompts like `move +x` or `close gripper` may be out of distribution. Use this to inspect behavior, not to claim official benchmark success.

In [ ]:
# @title Run One-Chunk Plaintext Prompt Probe

import os
import subprocess
from pathlib import Path

probe_out = LOCAL_REPO / "outputs/probes" / PROBE_SUITE / f"task_{PROBE_TASK_ID}"
probe_out.mkdir(parents=True, exist_ok=True)

probe_code = r"""
import json
import os
from pathlib import Path

import cv2
import numpy as np
import torch

from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.configs import LiberoEnv
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.factory import make_policy, make_pre_post_processors
from lerobot.scripts.lerobot_eval import preprocess_observation

suite = os.environ["PROBE_SUITE"]
task_id = int(os.environ["PROBE_TASK_ID"])
prompt = os.environ["PROBE_LANGUAGE"]
out_dir = Path(os.environ["PROBE_OUT"])
out_dir.mkdir(parents=True, exist_ok=True)

policy_cfg = PreTrainedConfig.from_pretrained(
    "lerobot/pi05_libero_finetuned",
    cache_dir=os.environ["HF_HUB_CACHE"],
    local_files_only=os.environ.get("HF_HUB_OFFLINE") == "1",
)
policy_cfg.device = "cuda"
policy_cfg.dtype = "bfloat16"
policy_cfg.compile_model = False
policy_cfg.gradient_checkpointing = False
policy_cfg.n_action_steps = 10

env_cfg = LiberoEnv(task=suite, task_ids=[task_id])
envs = make_env(env_cfg, n_envs=1, use_async_envs=False)
env = envs[suite][task_id]
policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg, rename_map={})
policy.eval()

preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=policy_cfg.pretrained_path,
    preprocessor_overrides={
        "device_processor": {"device": str(policy.config.device)},
        "rename_observations_processor": {"rename_map": {}},
    },
)
env_preprocessor, _env_postprocessor = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)

obs, info = env.reset(seed=[int(os.environ.get("PROBE_SEED", "1000"))])
raw_render = env.envs[0].render() if hasattr(env, "envs") else env.call("render")[0]
cv2.imwrite(str(out_dir / "render.png"), cv2.cvtColor(raw_render, cv2.COLOR_RGB2BGR))

obs = preprocess_observation(obs)
obs["task"] = [prompt]
for key, value in obs.items():
    if key.startswith("observation.images."):
        arr = value[0]
        if hasattr(arr, "detach"):
            arr = arr.detach().cpu().numpy()
        if arr.shape[0] in (1, 3):
            arr = np.moveaxis(arr, 0, -1)
        if arr.max() <= 1.5:
            arr = np.clip(arr, 0, 1) * 255
        cv2.imwrite(str(out_dir / f"{key.replace('.', '_')}.png"), cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_RGB2BGR))

batch = env_preprocessor(obs)
batch = preprocessor(batch)
with torch.inference_mode():
    actions = policy.predict_action_chunk(batch).detach().cpu().float()[0]

payload = {
    "suite": suite,
    "task_id": task_id,
    "prompt": prompt,
    "action_shape": list(actions.shape),
    "first_10_actions": actions[:10].tolist(),
    "mean_abs_per_dim": actions.abs().mean(dim=0).tolist(),
}
(out_dir / "action_chunk.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2)[:4000])
env.close()
"""

env = os.environ.copy()
env.update({
    "PROBE_SUITE": PROBE_SUITE,
    "PROBE_TASK_ID": str(PROBE_TASK_ID),
    "PROBE_LANGUAGE": PROBE_LANGUAGE,
    "PROBE_SEED": str(PROBE_SEED),
    "PROBE_OUT": str(probe_out),
})
subprocess.run([str(PYTHON), "-c", probe_code], cwd=LOCAL_REPO, env=env, check=True)
rsync(LOCAL_REPO / "outputs", DRIVE_OUTPUTS)
print("Probe saved to:", probe_out)

## Run All Checklist

Before sharing:

1. Save a copy of this notebook in the restricted Drive folder.
2. Share the notebook only with `programmer908@gmail.com` and other approved accounts.
3. Share `DRIVE_ROOT` only with the same approved accounts.
4. For first cache fill, keep `CACHE_TRANSFER_MODE="archive"`, set `ALLOW_AUTH_REFRESH=True`, and store `HF_TOKEN` only in private Colab Secrets.
5. After `archives/hf_home.tar` exists, leave `ALLOW_AUTH_REFRESH=True` or set it to `False`; the archive path is used first unless `FORCE_AUTH_REFRESH=True`.
6. Set Runtime -> Change runtime type -> GPU -> L4 when available. Keep `REQUIRED_GPU="Any"` if Colab gives you a T4 and you still want a smoke test.
7. Runtime -> Run all.

Outputs are persisted back to `DRIVE_ROOT/outputs` after every run, including rollout videos, diagnostic videos, chunk matrices, activation heatmaps, and individual layer graphs. Model/assets caches are persisted as tar archives under `DRIVE_ROOT/archives` so future sessions start faster.